In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *

In [2]:
# Load edge-df
with open("data_and_plots/dfs_of_magnetic_edge_information_0p5_zero-magmom_threshold.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("data_and_plots/df_stable_and_unique_MP_db.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "data_and_plots"

In [3]:
# The automatically determined oxidation states might be "unphysical" in many cases, e.g., that some ions are not expected to display magnetic order
# Nevertheless, we are showing the results as determined by the automated BVA
custom_sort_by_electron_configuration = [ 
    ("W", 6),
    ("V", 4),  ("Cr", 5), ("Mo", 5),
    ("V", 3), ("Cr", 4), ("Mo", 4), ("W", 4),  ("Re", 5), ("Os", 6),
    ("V", 2), ("Cr", 3), ("Mn", 4), ("Ru", 5), ("Mo", 3), ("Os", 5),
    ("Cr", 2), ("Mn", 3), ("Fe", 4),
    ("Ru", 4), ("Ir", 5),
    ("Mn", 2), ("Fe", 3), ("Co", 4),
    ("Rh", 4), ("Ir", 4),
    ("Fe", 2), ("Co", 3), ("Ni", 4),
    ("Co", 2), ("Ni", 3),
    ("Co", 1), ("Ni", 2),
    ("Ni", 1), ("Cu", 2)
]

In [4]:
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))

    ang_df["bond_angle_strict_180"] = ang_df["mag_ligand_mag_angle"].apply(lambda ls: len(ls) == 1 and round(ls[0], 4) == 180.0)
    ang_df["bond_angle_greater_150"] = ang_df["mag_ligand_mag_angle"].apply(lambda ls: len(ls) == 1 and round(ls[0], 4) > 150.0)


In [ ]:
write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"greater {ba_threshold}° bond angles with TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"bond_angle_greater_{ba_threshold}"]]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

#ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
#numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

greater 150° bond angles with TM octahedra at both nodes
{'afm': {('Fe', 3, 'Fe', 3): 140.0, ('Fe', 3, 'Mo', 5): 12.0, ('Mo', 5, 'Fe', 3): 12.0, ('V', 3, 'V', 3): 132.0, ('Cr', 3, 'Cr', 3): 252.0, ('Mn', 4, 'Mn', 4): 180.0, ('Co', 2, 'Co', 2): 108.0, ('Co', 4, 'Co', 4): 32.0, ('Ni', 2, 'Ni', 2): 260.0, ('Cr', 4, 'Mo', 4): 16.0, ('Mo', 4, 'Cr', 4): 16.0, ('Mn', 3, 'Mn', 3): 116.0, ('Fe', 3, 'Os', 5): 16.0, ('Os', 5, 'Fe', 3): 16.0, ('Fe', 2, 'Fe', 2): 104.0, ('Mn', 3, 'Mn', 2): 40.0, ('Mn', 2, 'Mn', 3): 40.0, ('Mn', 2, 'Mn', 2): 196.0, ('Co', 3, 'Co', 3): 92.0, ('Ni', 3, 'Ni', 3): 20.0, ('V', 2, 'V', 2): 40.0, ('Cu', 2, 'Ni', 2): 24.0, ('Ni', 2, 'Cu', 2): 24.0, ('V', 4, 'V', 4): 64.0, ('Fe', 4, 'Fe', 4): 8.0, ('Mn', 2, 'Ru', 5): 30.0, ('Ru', 5, 'Mn', 2): 30.0, ('Mn', 4, 'Cr', 4): 8.0, ('Cr', 4, 'Mn', 4): 8.0, ('Co', 3, 'Ir', 4): 8.0, ('Ir', 4, 'Co', 3): 8.0, ('V', 4, 'Mo', 4): 8.0, ('Mo', 4, 'Mo', 4): 4.0, ('Mo', 4, 'V', 4): 8.0, ('Cr', 2, 'W', 6): 40.0, ('W', 6, 'Cr', 2): 40.0, ('V', 3

In [ ]:
# Repeat analysis considering only oxygen edges

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"greater {ba_threshold}° bond angles with TM octahedra at both nodes only oxygen edges"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"bond_angle_greater_{ba_threshold}"]]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

#ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
#numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

greater 150° bond angles with TM octahedra at both nodes only oxygen edges
{'afm': {('Fe', 3, 'Fe', 3): 140.0, ('Fe', 3, 'Mo', 5): 12.0, ('Mo', 5, 'Fe', 3): 12.0, ('V', 3, 'V', 3): 120.0, ('Cr', 3, 'Cr', 3): 248.0, ('Mn', 4, 'Mn', 4): 180.0, ('Co', 2, 'Co', 2): 108.0, ('Co', 4, 'Co', 4): 32.0, ('Ni', 2, 'Ni', 2): 236.0, ('Cr', 4, 'Mo', 4): 16.0, ('Mo', 4, 'Cr', 4): 16.0, ('Mn', 3, 'Mn', 3): 92.0, ('Fe', 3, 'Os', 5): 16.0, ('Os', 5, 'Fe', 3): 16.0, ('Fe', 2, 'Fe', 2): 88.0, ('Mn', 3, 'Mn', 2): 40.0, ('Mn', 2, 'Mn', 3): 40.0, ('Mn', 2, 'Mn', 2): 192.0, ('Co', 3, 'Co', 3): 92.0, ('Ni', 3, 'Ni', 3): 20.0, ('V', 2, 'V', 2): 40.0, ('Cu', 2, 'Ni', 2): 24.0, ('Ni', 2, 'Cu', 2): 24.0, ('V', 4, 'V', 4): 48.0, ('Fe', 4, 'Fe', 4): 8.0, ('Mn', 2, 'Ru', 5): 30.0, ('Ru', 5, 'Mn', 2): 30.0, ('Mn', 4, 'Cr', 4): 8.0, ('Cr', 4, 'Mn', 4): 8.0, ('Co', 3, 'Ir', 4): 8.0, ('Ir', 4, 'Co', 3): 8.0, ('V', 4, 'Mo', 4): 8.0, ('Mo', 4, 'Mo', 4): 4.0, ('Mo', 4, 'V', 4): 8.0, ('Cr', 2, 'W', 6): 40.0, ('W', 6, 'Cr', 2

In [ ]:
# Repeat analysis only considering structures with one magnetic ion to eliminate possible mixed valence issue a bit

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"greater {ba_threshold}° bond angles with TM octahedra at both nodes only one mag. ion per structure"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    if len(set(ang_df["site_element"])) == 1 and len(set(ang_df["site_oxidation"])) == 1:
        subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
        subdf = subdf.loc[subdf[f"bond_angle_greater_{ba_threshold}"]]
        subdf = subdf.loc[(subdf["site_ce"]=="O:6")
                & (subdf["site_to_ce"]=="O:6")]

        if not subdf.empty:
            n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
            for mag_type, condition in zip(["afm", "fm", ],
                                            [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
                type_df = subdf.loc[condition]
                if not type_df.empty:
                    abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                    for k, v in abs_ion_pairs.items():
                        # Assert no detected ions are missing
                        try:
                            assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                            assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                        except AssertionError as e:
                            missed_ions.add(str(e))
                        if k in occus[mag_type]:
                            occus[mag_type][k] += v
                        else:
                            occus[mag_type][k] = v

                        if k in num_structures:
                            num_structures[k].add(md_id)
                        else:
                            num_structures[k] = {md_id}


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

#ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

greater 150° bond angles with TM octahedra at both nodes only one mag. ion per structure
ratio of -1.0334237554869496 in ('Cr', 3, 'Cr', 3)
ratio of 1.4548448600085102 in ('Fe', 4, 'Fe', 4)
ratio of -1.0791812460476249 in ('Co', 2, 'Co', 2)
FM to AFM ratio  :
('Mn', 2, 'Mn', 2) 0.16 -0.81 14
('Fe', 4, 'Fe', 4) 28.5 1.45 11
('V', 3, 'V', 3) 1.09 0.04 19
('Co', 4, 'Co', 4) 0.5 -0.3 3
('Fe', 2, 'Fe', 2) 0.35 -0.46 16
('Mn', 4, 'Mn', 4) 1.82 0.26 28
('Mn', 3, 'Mn', 3) 2.83 0.45 44
('Co', 3, 'Co', 3) 0.38 -0.41 7
('V', 4, 'V', 4) 2.41 0.38 17
('Cr', 3, 'Cr', 3) 0.09 -1.03 17
('Ni', 2, 'Ni', 2) 0.19 -0.72 17
('Ni', 3, 'Ni', 3) 5.5 0.74 6
('Co', 2, 'Co', 2) 0.08 -1.08 8
('Fe', 3, 'Fe', 3) 0.18 -0.74 15
('V', 2, 'V', 2) 4.0 0.6 2
('Cr', 4, 'Cr', 4) 0.89 -0.05 3
8 16 0.5
Num structures:  227
missed:  set()
